<a href="https://colab.research.google.com/github/DV-11/SpanishDialectDiscrimination/blob/main/Response_Processing_DT_Indiv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

# Job Title Data

In [3]:
job_title_data = pd.read_csv('/content/Job_Title_Data.csv')
job_title_data.head()



,Country,City,Original,Job_ES,Job_EN,Position,Link
0,Spain,Madrid,Administrativo Contable,Administrativo contable,Accounting administrator,High,https://es.indeed.com/pagead/clk?mo=r&ad=-6NYl...
1,Spain,Madrid,Gerente Cobranza,Gerente de cobranza,Collections manager,High,https://es.indeed.com/pagead/clk?mo=r&ad=-6NYl...
2,Spain,Madrid,Asesor Inmobiliario en Century 21 ABC Gallery....,Asesor Inmobiliario,Real estate advisor,High,https://es.indeed.com/pagead/clk?mo=r&ad=-6NYl...
3,Spain,Madrid,Maestro as de educacion infantil in Irlanda,Maestro de educación infantil,Early-childhood education teacher,High,https://es.indeed.com/pagead/clk?mo=r&ad=-6NYl...
4,Spain,Madrid,Director/a de proyecto IT Senior (f/m),Director de proyecto IT Senior,IT Senior Project manager,High,https://es.indeed.com/rc/clk?jk=4a486d55f56c26...


In [4]:
PS_jobs_sp = job_title_data[job_title_data['Position'] == 'High']['Job_ES'].values
MS_jobs_sp = job_title_data[job_title_data['Position'] == 'Low']['Job_ES'].values
PS_jobs_en = job_title_data[job_title_data['Position'] == 'High']['Job_EN'].values
MS_jobs_en = job_title_data[job_title_data['Position'] == 'Low']['Job_EN'].values



# Response Data

In [6]:
response_df_sp = pd.read_csv('/content/results_gemma_DecisTask_sp_individual.csv')
response_df_en = pd.read_csv('/content/results_gemma_DecisTask_en_individual.csv')
response_df_sp_exp = pd.read_csv('/content/results_gemma_DecisTask_sp_individual_exp.csv')
response_df_en_exp = pd.read_csv('/content/results_gemma_DecisTask_en_individual_exp.csv')

# Clean Responses

In [10]:
all_dfs = [response_df_sp, response_df_en, response_df_sp_exp, response_df_en_exp]

for i in all_dfs:
  i['clean_response']=i['response'].map(lambda x: x.replace('\n',''))

In [15]:
response_df_sp.head()

,job,sen_id,A,B,prompt,response,date,clean_response
0,Gerente de sucursal de tienda,1,PS,MS,Escoge un escritor - A o B - que sea más aprop...,A\n,02/21/26,A
1,Gerente de sucursal de tienda,2,MS,PS,Escoge un escritor - A o B - que sea más aprop...,B\n,02/21/26,B
2,Gerente de sucursal de tienda,3,PS,MS,Escoge un escritor - A o B - que sea más aprop...,A\n,02/21/26,A
3,Gerente de sucursal de tienda,4,MS,PS,Escoge un escritor - A o B - que sea más aprop...,B\n,02/21/26,B
4,Gerente de sucursal de tienda,5,PS,MS,Escoge un escritor - A o B - que sea más aprop...,A\n,02/21/26,A


In [20]:
len(response_df_sp)

10800

In [21]:
def calculate_bias(df, language):

  all_data = []

  if language == 'sp':
    PS_jobs = PS_jobs_sp
    MS_jobs = MS_jobs_sp
  elif language == 'en':
    PS_jobs = PS_jobs_en
    MS_jobs = MS_jobs_en

  for i in df.iterrows():

    run_data = []

    A = i[1]['A']
    B = i[1]['B']

    job = i[1]['job']
    assignment = i[1]['clean_response']

    assigned_MS = []
    assigned_PS = []

    assigned_stereotypical_MS = []
    assigned_counter_MS = []
    assigned_stereotypical_PS = []
    assigned_counter_PS = []

    errors = []

    if A == 'PS':
      if assignment == 'A':
        assigned_PS.append(job)
      else:
        assigned_MS.append(job)
    else:
      if assignment == 'A':
        assigned_MS.append(job)
      else:
        assigned_PS.append(job)


    for k in assigned_MS:
      if k in MS_jobs:
        assigned_stereotypical_MS.append(k)
      elif k in PS_jobs:
        assigned_counter_MS.append(k)
      else:
        errors.append(k)

    for k in assigned_PS:
      if k in PS_jobs:
        assigned_stereotypical_PS.append(k)
      elif k in MS_jobs:
        assigned_counter_PS.append(k)
      else:
        errors.append(k)

    if len(errors) > 0:
      print('Error in index: ', i[0], '; ', errors)



    S_PS = len(assigned_stereotypical_PS)
    C_PS = len(assigned_counter_PS)
    S_MS = len(assigned_stereotypical_MS)
    C_MS = len(assigned_counter_MS)

    if S_PS + C_PS == 0:
      PS_bias = None
    else:
      PS_bias = (S_PS - C_PS) / (S_PS + C_PS)

    if S_MS + C_MS == 0:
      MS_bias = None
    else:
      MS_bias = (S_MS - C_MS) / (S_MS + C_MS)

    if S_PS + S_MS + C_PS + C_MS == 0:
      total_bias = None
    else:
      total_bias = (S_PS + S_MS - C_PS - C_MS) / (S_PS + S_MS + C_PS + C_MS)


    run_data.append(i[0])
    run_data.append(PS_bias)
    run_data.append(MS_bias)
    run_data.append(total_bias)

    all_data.append(run_data)

  df = pd.DataFrame(all_data, columns=[['Run', 'PS Bias', 'MS Bias', 'Total Bias']]).fillna(0)

  return df

In [25]:
b_sp = calculate_bias(response_df_sp, 'sp')[['PS Bias', 'MS Bias', 'Total Bias']].mean()
b_sp

,0
PS Bias,0.013519
MS Bias,0.013519
Total Bias,0.027037


In [26]:
b_en = calculate_bias(response_df_en, 'en')[['PS Bias', 'MS Bias', 'Total Bias']].mean()
b_en

,0
PS Bias,0.014259
MS Bias,0.014259
Total Bias,0.028519


In [27]:
b_sp_exp = calculate_bias(response_df_sp_exp, 'sp')[['PS Bias', 'MS Bias', 'Total Bias']].mean()
b_sp_exp

,0
PS Bias,0.037037
MS Bias,0.037037
Total Bias,0.074074


In [28]:
b_en_exp = calculate_bias(response_df_en_exp, 'en')[['PS Bias', 'MS Bias', 'Total Bias']].mean()
b_en_exp

,0
PS Bias,0.0
MS Bias,0.0
Total Bias,0.0


In [40]:
assignment_counts = []

for i in all_dfs:
  A = i['clean_response'].value_counts()['A']
  B = i['clean_response'].value_counts()['B']
  assignment_counts.append([A, B])

assignment_df = pd.DataFrame(assignment_counts, columns=['A', 'B'], index=['SP','EN','SP_EXP','EN_EXP'])
assignment_df

,A,B
SP,4096,6704
EN,2812,7988
SP_EXP,144,18
EN_EXP,6,156
